# Filter 1.0
## FineWeb Cleaning and Filtering Notebook

This notebook focuses on initial filtering experiments for one FineWeb parquet file.

It is organised into three parts:
1. setup and derived variables
2. Week 2 initial cleaning
3. Week 3 follow-up work

In [2]:
import os
import time
import pandas as pd

In [3]:
def load_parquet_file(base_dir, file_name=None):
    if not os.path.exists(base_dir):
        raise FileNotFoundError(f"Base directory does not exist: {base_dir}")

    parquet_files = sorted([
        f for f in os.listdir(base_dir)
        if f.endswith(".parquet")
    ])

    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in {base_dir}")

    if file_name is None:
        file_name = parquet_files[0]

    file_path = os.path.join(base_dir, file_name)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Parquet file does not exist: {file_path}")

    print("Loading file:", file_path)
    df = pd.read_parquet(file_path)
    return df, file_path, parquet_files

In [4]:
base_dir = "/home/jovyan/data/fineweb"
file_name = "004_00018.parquet"   # change when needed

start = time.time()
df, file_path, parquet_files = load_parquet_file(base_dir, file_name)
read_time = time.time() - start

print("Loaded file:", file_path)
print("Shape:", df.shape)
print("Read parquet time (seconds):", read_time)

Loading file: /home/jovyan/data/fineweb/004_00018.parquet
Loaded file: /home/jovyan/data/fineweb/004_00018.parquet
Shape: (172447, 9)
Read parquet time (seconds): 0.8417215347290039


In [5]:
df["text_length"] = df["text"].fillna("").str.len()
df["estimated_token_count"] = df["text"].fillna("").str.split().str.len()

print(df[["text_length", "estimated_token_count"]].head())

   text_length  estimated_token_count
0          899                    147
1        15150                   2209
2         2512                    481
3         2141                    396
4         6327                   1096


In [6]:
print("Columns:", df.columns.tolist())
print(df.dtypes)

Columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'text_length', 'estimated_token_count']
text                         str
id                           str
dump                         str
url                          str
date                         str
file_path                    str
language                     str
language_score           float64
token_count                int64
text_length                int64
estimated_token_count      int64
dtype: object


## Part 2: Week 2 Initial Cleaning

This part reproduces the initial filtering pipeline developed during Week 2.
The goal is to apply a few simple and general rules before exploring more specific cleaning ideas.

In [10]:
##remove non-english 
english_df = df[df["language"] == "en"]
print("Original shape:", df.shape)
print("English-only shape:", english_df.shape)

Original shape: (172447, 11)
English-only shape: (172447, 11)


In [11]:
##keep high-level language confidence
high_conf_df = english_df[english_df["language_score"] >= 0.9]
print("English-only shape:", english_df.shape)
print("English + high language_score shape:", high_conf_df.shape)

English-only shape: (172447, 11)
English + high language_score shape: (148399, 11)


In [12]:
##remove short texts
length_filtered_df = high_conf_df[high_conf_df["text_length"] >= 200]
print("Before text length filtering:", high_conf_df.shape)
print("After text length filtering:", length_filtered_df.shape)

Before text length filtering: (148399, 11)
After text length filtering: (148391, 11)


In [13]:
##remove duplicated URL
duplicate_url_count = length_filtered_df["url"].duplicated().sum()
print("Number of duplicated URLs:", duplicate_url_count)
dedup_url_df = length_filtered_df.drop_duplicates(subset=["url"])
print("Before URL deduplication:", length_filtered_df.shape)
print("After URL deduplication:", dedup_url_df.shape)

Number of duplicated URLs: 13
Before URL deduplication: (148391, 11)
After URL deduplication: (148378, 11)


In [14]:
##remove duplicated text
duplicate_text_count = length_filtered_df["text"].duplicated().sum()
print("Number of duplicated text samples:", duplicate_text_count)
dedup_text_df = length_filtered_df.drop_duplicates(subset=["text"])
print("Before text deduplication:", length_filtered_df.shape)
print("After text deduplication:", dedup_text_df.shape)

Number of duplicated text samples: 0
Before text deduplication: (148391, 11)
After text deduplication: (148391, 11)


In [15]:
##Week 2 filtering summary
filter_pipeline_summary = pd.DataFrame([
    {"step": "original", "rows": len(df)},
    {"step": "english_only", "rows": len(english_df)},
    {"step": "high_language_score", "rows": len(high_conf_df)},
    {"step": "text_length_filter", "rows": len(length_filtered_df)},
])

filter_pipeline_summary["rows_removed"] = (
    filter_pipeline_summary["rows"].shift(1) - filter_pipeline_summary["rows"]
)
filter_pipeline_summary.loc[0, "rows_removed"] = 0

filter_pipeline_summary

,step,rows,rows_removed
0,original,172447,0.0
1,english_only,172447,0.0
2,high_language_score,148399,24048.0
3,text_length_filter,148391,8.0


In [16]:
##Duplicate summary
duplicate_summary = pd.DataFrame([
    {
        "check": "url_duplicates",
        "before_rows": len(length_filtered_df),
        "duplicates_found": duplicate_url_count,
        "after_dedup_rows": len(dedup_url_df)
    },
    {
        "check": "text_duplicates",
        "before_rows": len(length_filtered_df),
        "duplicates_found": duplicate_text_count,
        "after_dedup_rows": len(dedup_text_df)
    },
])

duplicate_summary

,check,before_rows,duplicates_found,after_dedup_rows
0,url_duplicates,148391,13,148378
1,text_duplicates,148391,0,148391


## Part 3: Week 3 Follow-up Work

This part continues the previous exploration by:
- checking low-information content
- comparing filtering ideas across the team
- recording IO timing
- exporting filtered samples and summaries

In [18]:
##Low-information keyword check
low_info_keywords = [
    "main menu",
    "disclaimer",
    "privacy policy",
    "cookie policy",
    "terms of use",
    "all rights reserved",
    "sign up",
    "log in"
]

low_info_mask = dedup_text_df["text"].fillna("").str.lower().apply(
    lambda x: any(keyword in x for keyword in low_info_keywords)
)

low_info_df = dedup_text_df[low_info_mask]

print("Rows matching low-information keywords:", low_info_df.shape)
low_info_df[["text", "url"]].head(20)

Rows matching low-information keywords: (7197, 11)


,text,url
12,Online lottery is a game of chance where playe...,https://www.ubuntumade.com/how-to-find-a-reput...
32,Slot Casino Australia\nBy choosing the right m...,https://www.virginiahistorichomes.org/slot-cas...
100,"When it comes to airports, designers in recent...",https://yourmileagemayvary.com/2025/05/16/airp...
130,Legal Australian Online Pokies\nThis bonus is ...,http://mavrealtors.com/internet-gambling/are-a...
138,Variegated Red Twig Dogwood\nCornus alba 'Eleg...,http://plants.rutgersln.com/12150011/Plant/705...
163,"Isn’t it funny how Republicans talk tough, but...",http://www.gregdewar.com/2005/04/when_we_want_...
169,Gold Ideals and Management\nCommodities / Gold...,http://www.marketoracle.co.uk/Article50450.html
183,"I’m Jasmine, Women’s Holistic Health & Nutriti...",http://www.thewholesomeheart.com/
208,Be the first to become informed about the AFA’...,https://afa.net/the-stand/culture/2015/05/past...
209,Be the first to become informed about the AFA’...,https://afa.net/the-stand/family/2019/05/a-vic...


In [19]:
##Suspicious URL pattern check
suspicious_url_keywords = [
    "ads",
    "advert",
    "utm_",
    "tracking",
    "click",
    "redirect"
]

suspicious_url_mask = dedup_text_df["url"].fillna("").str.lower().apply(
    lambda x: any(keyword in x for keyword in suspicious_url_keywords)
)

suspicious_url_df = dedup_text_df[suspicious_url_mask]

print("Rows matching suspicious URL patterns:", suspicious_url_df.shape)
suspicious_url_df[["url", "text"]].head(20)

Rows matching suspicious URL patterns: (2065, 11)


,url,text
59,https://www.windowsbasics.com/2024/05/how-to-d...,Microsoft has made a lot of changes to Windows...
140,http://righthearmobiletesting.com/free-pokies-...,Free Pokies Downloads Games\nThere are 46 stat...
279,https://auto.economictimes.indiatimes.com/news...,"Commodity prices, recovery in construction act..."
316,https://blog.bernina.com/en/2015/07/who-needs-...,I set myself a little challenge recently: NO M...
410,https://classicalconversations.com/blog/dads-g...,Are you confused about classical education? Ar...
474,https://direct.mit.edu/pvar/article-abstract/9...,This paper reports on an investigation into th...
752,https://lawfitz.com/blog/kent-man-pleads-guilt...,A Kent man pleaded guilty to the hit-and-run d...
753,https://layarkaca21.sbs/wp-content/uploads/202...,Slot Fruit Super Nova By Evoplay Demo Free Pla...
826,https://myclickjournal.com/europe-2-week-itine...,how I traveled to 6 countries in 16 days!\nAs ...
934,https://proceedings.aiche.org/conferences/aich...,2015 AIChE Annual Meeting Proceedings\n(47f) N...


In [23]:
## Check very long text samples
very_long_df = dedup_text_df[dedup_text_df["text_length"] > 10000]

print("Rows with text_length > 10000:", very_long_df.shape)
very_long_df[["text", "url", "text_length"]].head(20)

Rows with text_length > 10000: (9333, 11)


,text,url,text_length
1,"The Art of Effective Board Reporting\nSep 5, 2...",https://www.traact.com/blog/the-art-of-effecti...,15150
7,Having a baby is often described as an excitin...,https://www.treatmyocd.com/blog/postpartum-int...,14174
29,Vikings Offensive Coordinator Norv Turner\nQ: ...,https://www.vikings.com/news/coordinators-addr...,17252
78,Navigating the modern business environment req...,https://www.youngurbanproject.com/scope-of-bus...,19356
84,Jonas Lund applies the methods of quantificati...,https://www.zerodeux.fr/en/interviews-en/jonas...,15653
90,How automated litigation support is a game-cha...,https://xapien.com/insights/litigation-support...,10694
98,There’s a lot to think about when you’re headi...,https://yourcareerplace.com/tag/impressions/,14987
131,Bally Casino Games NJ and PA. Users may have a...,http://mei-hongqi-ly.com/optimal-blackjack-tac...,11127
148,Title on the grown area site demonstrably show...,http://tsttransportation.com/adultspace-assess...,14387
194,Hendrik Van Der Dussen\nFeb 07' 1989\nThe Exhi...,https://8lete.life/stories/cricketer-rassie-va...,13703


In [24]:
## Check rows with high non-alphanumeric character ratio
dedup_text_df["non_alnum_ratio"] = dedup_text_df["text"].fillna("").apply(
    lambda x: sum(1 for c in x if not c.isalnum() and not c.isspace()) / max(len(x), 1)
)

weird_char_df = dedup_text_df[dedup_text_df["non_alnum_ratio"] > 0.3]

print("Rows with high non-alphanumeric ratio:", weird_char_df.shape)
weird_char_df[["text", "url", "non_alnum_ratio"]].head(20)

Rows with high non-alphanumeric ratio: (0, 12)


,text,url,non_alnum_ratio


In [25]:
## Check repeated line ratio inside each text sample
def repeated_line_ratio(text):
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    if not lines:
        return 0
    unique_lines = set(lines)
    return 1 - len(unique_lines) / len(lines)

dedup_text_df["repeated_line_ratio"] = dedup_text_df["text"].fillna("").apply(repeated_line_ratio)

repeated_line_df = dedup_text_df[dedup_text_df["repeated_line_ratio"] > 0.3]

print("Rows with repeated line ratio > 0.3:", repeated_line_df.shape)
repeated_line_df[["text", "url", "repeated_line_ratio"]].head(20)

Rows with repeated line ratio > 0.3: (0, 13)


,text,url,repeated_line_ratio


In [26]:
## Check HTML-like or code-like residual patterns
html_code_keywords = [
    "<div", "<span", "<script", "</a>", "function(", "{", "}", "var ", "const "
]

html_code_mask = dedup_text_df["text"].fillna("").str.lower().apply(
    lambda x: any(keyword in x for keyword in html_code_keywords)
)

html_code_df = dedup_text_df[html_code_mask]

print("Rows matching HTML/code-like patterns:", html_code_df.shape)
html_code_df[["text", "url"]].head(20)

Rows matching HTML/code-like patterns: (292, 13)


,text,url
159,Image from: zimbio.com\nReview of the Day\nR.I...,http://www.football-news-views.co.uk/football-...
480,Thank you for helping us improve the quality o...,https://docs.unity3d.com/2019.2/Documentation/...
1837,"Download the mp3 of this episode, or read the ...",https://www.smartlinx.com/resources/podcasts/l...
2254,"Baramulla : The Deputy Commissioner, State Tax...",https://cnionline.in/b2v5-dc-state-taxes-liste...
2975,"Tuesday, 15 October 2013\n125 Basic C# Intervi...",https://theprofessionalspoint.blogspot.com/201...
3522,"And, like most, he’s a former Episcopalian:\nA...",https://www.patheos.com/blogs/deaconsbench/201...
3842,Kochubey Mansion on Konnogvardeyskiy Bulvar\nT...,http://www.saint-petersburg.com/mansions/kochu...
3846,"Free Legit & Rage Cheats | Glow, Fake Lag, No ...",http://ximangtanquang.com.vn/free-legit-rage-c...
4804,Claudio Abbado | |\nBorn | |\nDied | 20 Januar...,https://wikimili.com/en/Claudio_Abbado
5020,"Eugenio Brito Honorato\nPainter, muralist, cer...",https://www.eugeniobrito.cl/en/biografia/


In [27]:
## Check domain frequency distribution
dedup_text_df["domain"] = dedup_text_df["url"].fillna("").str.extract(r"https?://([^/]+)")

domain_counts = dedup_text_df["domain"].value_counts().reset_index()
domain_counts.columns = ["domain", "count"]

print(domain_counts.head(20))

                         domain  count
0      plandisney.disney.go.com     51
1           www.telegraph.co.uk     46
2       pubmed.ncbi.nlm.nih.gov     43
3             www.ibtimes.co.uk     43
4               www.cbsnews.com     41
5               www.latimes.com     38
6          www.sciencedaily.com     38
7   timesofindia.indiatimes.com     37
8                 www.prweb.com     36
9    publications.parliament.uk     33
10              www.foxnews.com     32
11             www.gofundme.com     32
12            www.techradar.com     31
13            www.csmonitor.com     29
14            www.aljazeera.com     28
15             www.jalopnik.com     27
16        www.digitaltrends.com     27
17             forum.trains.com     26
18       www.themoscowtimes.com     25
19    www.business-standard.com     25


In [28]:
## Inspect samples from the most frequent domain
top_domain = domain_counts.iloc[0]["domain"]
top_domain_df = dedup_text_df[dedup_text_df["domain"] == top_domain]

print("Top domain:", top_domain)
top_domain_df[["url", "text"]].head(10)

Top domain: plandisney.disney.go.com


,url,text
8103,https://plandisney.disney.go.com/question/book...,With Advance Dining Reservations open 180 days...
8104,https://plandisney.disney.go.com/question/ever...,"Hi, Jean! If it were up to me, I'd choose lunc..."
9839,https://plandisney.disney.go.com/question/inpu...,"Hi there, Gloria!\nWelcome back to planDisney ..."
11577,https://plandisney.disney.go.com/question/neig...,Thanks so much for sending this great question...
13343,https://plandisney.disney.go.com/question/wife...,Hi Ben! Thanks for visiting planDisney.\nCongr...
15139,https://plandisney.disney.go.com/question/delu...,"Welcome aboard, Dona!\nI’m so happy to connect..."
26093,https://plandisney.disney.go.com/question/im-p...,"Welcome back to planDisney, Cali! It's always ..."
26094,https://plandisney.disney.go.com/question/im-p...,Wow - a surprise Christmas vacation! I know yo...
26095,https://plandisney.disney.go.com/question/purc...,Mickey's Not-So-Scary Halloween Party is one o...
27863,https://plandisney.disney.go.com/question/idea...,I can tell that you are super excited about yo...


In [30]:
## Measure parquet loading time
import time

base_dir = "/home/jovyan/data/fineweb"
file_name = "004_00018.parquet"   # change when needed

start = time.time()
df, file_path, parquet_files = load_parquet_file(base_dir, file_name)
read_time = time.time() - start

print("Loaded file:", file_path)
print("Shape:", df.shape)
print("Read parquet time (seconds):", read_time)

Loading file: /home/jovyan/data/fineweb/004_00018.parquet
Loaded file: /home/jovyan/data/fineweb/004_00018.parquet
Shape: (172447, 9)
Read parquet time (seconds): 0.7270047664642334


In [31]:
## Measure low-information content check time
start = time.time()

low_info_keywords = [
    "main menu",
    "disclaimer",
    "privacy policy",
    "cookie policy",
    "terms of use",
    "all rights reserved",
    "sign up",
    "log in"
]

low_info_mask = dedup_text_df["text"].fillna("").str.lower().apply(
    lambda x: any(keyword in x for keyword in low_info_keywords)
)

low_info_df = dedup_text_df[low_info_mask]

low_info_time = time.time() - start

print("Rows matching low-information keywords:", low_info_df.shape)
print("Low-information check time (seconds):", low_info_time)

Rows matching low-information keywords: (7197, 14)
Low-information check time (seconds): 6.318031549453735


In [37]:
## Measure suspicious URL pattern check time
start = time.time()

suspicious_url_keywords_timing = [
    "ads",
    "advert",
    "utm_",
    "tracking",
    "click",
    "redirect"
]

suspicious_url_mask_timing = dedup_text_df["url"].fillna("").str.lower().apply(
    lambda x: any(keyword in x for keyword in suspicious_url_keywords_timing)
)

suspicious_url_df_timing = dedup_text_df[suspicious_url_mask_timing]

suspicious_url_time = time.time() - start

print("Rows matching suspicious URL patterns (timing run):", suspicious_url_df_timing.shape)
print("Suspicious URL check time (seconds):", suspicious_url_time)

Rows matching suspicious URL patterns (timing run): (2065, 14)
Suspicious URL check time (seconds): 0.24226117134094238


In [38]:
## Measure very long text check time
start = time.time()

very_long_df_timing = dedup_text_df[dedup_text_df["text_length"] > 10000]

very_long_time = time.time() - start

print("Rows with text_length > 10000 (timing run):", very_long_df_timing.shape)
print("Very long text check time (seconds):", very_long_time)

Rows with text_length > 10000 (timing run): (9333, 14)
Very long text check time (seconds): 0.34515833854675293


In [39]:
## Measure non-alphanumeric ratio check time
start = time.time()

non_alnum_ratio_series_timing = dedup_text_df["text"].fillna("").apply(
    lambda x: sum(1 for c in x if not c.isalnum() and not c.isspace()) / max(len(x), 1)
)

weird_char_df_timing = dedup_text_df[non_alnum_ratio_series_timing > 0.3]

non_alnum_time = time.time() - start

print("Rows with high non-alphanumeric ratio (timing run):", weird_char_df_timing.shape)
print("Non-alphanumeric ratio check time (seconds):", non_alnum_time)

Rows with high non-alphanumeric ratio (timing run): (0, 14)
Non-alphanumeric ratio check time (seconds): 17.8100426197052


In [41]:
## Measure repeated line ratio check time
start = time.time()

def repeated_line_ratio_timing(text):
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    if not lines:
        return 0
    unique_lines = set(lines)
    return 1 - len(unique_lines) / len(lines)

repeated_line_ratio_series_timing = dedup_text_df["text"].fillna("").apply(repeated_line_ratio_timing)
repeated_line_df_timing = dedup_text_df[repeated_line_ratio_series_timing > 0.3]

repeated_line_time = time.time() - start

print("Rows with repeated line ratio > 0.3 (timing run):", repeated_line_df_timing.shape)
print("Repeated line ratio check time (seconds):", repeated_line_time)

Rows with repeated line ratio > 0.3 (timing run): (0, 14)
Repeated line ratio check time (seconds): 1.9200222492218018


In [34]:
## Measure HTML/code residual check time
import time

start = time.time()

html_code_keywords_timing = [
    "<div", "<span", "<script", "</a>", "function(", "{", "}", "var ", "const "
]

html_code_mask_timing = dedup_text_df["text"].fillna("").str.lower().apply(
    lambda x: any(keyword in x for keyword in html_code_keywords_timing)
)

html_code_df_timing = dedup_text_df[html_code_mask_timing]

html_code_time = time.time() - start

print("Rows matching HTML/code-like patterns (timing run):", html_code_df_timing.shape)
print("HTML/code check time (seconds):", html_code_time)

Rows matching HTML/code-like patterns (timing run): (292, 14)
HTML/code check time (seconds): 5.512956619262695


In [32]:
## Measure domain frequency check time
start = time.time()

dedup_text_df["domain"] = dedup_text_df["url"].fillna("").str.extract(r"https?://([^/]+)")

domain_counts = dedup_text_df["domain"].value_counts().reset_index()
domain_counts.columns = ["domain", "count"]

domain_frequency_time = time.time() - start

print(domain_counts.head(20))
print("Domain frequency check time (seconds):", domain_frequency_time)

                         domain  count
0      plandisney.disney.go.com     51
1           www.telegraph.co.uk     46
2       pubmed.ncbi.nlm.nih.gov     43
3             www.ibtimes.co.uk     43
4               www.cbsnews.com     41
5               www.latimes.com     38
6          www.sciencedaily.com     38
7   timesofindia.indiatimes.com     37
8                 www.prweb.com     36
9    publications.parliament.uk     33
10              www.foxnews.com     32
11             www.gofundme.com     32
12            www.techradar.com     31
13            www.csmonitor.com     29
14            www.aljazeera.com     28
15             www.jalopnik.com     27
16        www.digitaltrends.com     27
17             forum.trains.com     26
18       www.themoscowtimes.com     25
19    www.business-standard.com     25
Domain frequency check time (seconds): 0.3103759288787842


In [44]:
## Measure top-domain sample inspection time
start = time.time()

top_domain_timing = domain_counts.iloc[0]["domain"]
top_domain_df_timing = dedup_text_df[dedup_text_df["domain"] == top_domain_timing]
top_domain_preview_timing = top_domain_df_timing[["url", "text"]].head(10)

top_domain_inspection_time = time.time() - start

print("Top domain (timing run):", top_domain_timing)
print("Top-domain inspection time (seconds):", top_domain_inspection_time)


Top domain (timing run): plandisney.disney.go.com
Top-domain inspection time (seconds): 0.007103681564331055


In [35]:
## Export filtered sample from the current main filtered dataset
start = time.time()

preview_sample = dedup_text_df[["text"]].sample(
    n=min(50, len(dedup_text_df)),
    random_state=42
).copy()

preview_sample["text"] = (
    preview_sample["text"]
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .apply(lambda x: x[:150] + "..." if len(x) > 150 else x)
)

preview_sample.to_csv("004_00018.clean_sample50_detailed_preview150.csv", index=False)

export_time = time.time() - start
print("Export CSV time (seconds):", export_time)

Export CSV time (seconds): 0.023760557174682617


In [45]:
## Summarise Week 3 IO timing results
week3_io_timing_summary = pd.DataFrame([
    {"operation": "low_info_check", "seconds": low_info_time},
    {"operation": "suspicious_url_check", "seconds": suspicious_url_time},
    {"operation": "very_long_text_check", "seconds": very_long_time},
    {"operation": "non_alnum_ratio_check", "seconds": non_alnum_time},
    {"operation": "repeated_line_check", "seconds": repeated_line_time},
    {"operation": "html_code_check", "seconds": html_code_time},
    {"operation": "domain_frequency_check", "seconds": domain_frequency_time},
    {"operation": "top_domain_inspection", "seconds": top_domain_inspection_time},
])

week3_io_timing_summary

,operation,seconds
0,low_info_check,6.318032
1,suspicious_url_check,0.242261
2,very_long_text_check,0.345158
3,non_alnum_ratio_check,18.428292
4,repeated_line_check,1.920022
5,html_code_check,5.512957
6,domain_frequency_check,0.310376
7,top_domain_inspection,0.007104


In [46]:
## Export Week 3 outputs

if len(low_info_df) > 0:
    low_info_df[["text", "url"]].head(50).to_csv(
        "004_00018.low_info_keyword_hits.csv", index=False
    )

if len(suspicious_url_df) > 0:
    suspicious_url_df[["url", "text"]].head(50).to_csv(
        "004_00018.suspicious_url_hits.csv", index=False
    )

if len(very_long_df) > 0:
    very_long_df[["text", "url", "text_length"]].head(50).to_csv(
        "004_00018.very_long_sample50.csv", index=False
    )

if len(weird_char_df) > 0:
    weird_char_df[["text", "url", "non_alnum_ratio"]].head(50).to_csv(
        "004_00018.weird_char_sample50.csv", index=False
    )

if len(repeated_line_df) > 0:
    repeated_line_df[["text", "url", "repeated_line_ratio"]].head(50).to_csv(
        "004_00018.repeated_line_sample50.csv", index=False
    )

if len(html_code_df) > 0:
    html_code_df[["text", "url"]].head(50).to_csv(
        "004_00018.html_code_sample50.csv", index=False
    )

domain_counts.to_csv("004_00018.domain_counts.csv", index=False)

if len(top_domain_df) > 0:
    top_domain_df[["url", "text"]].head(50).to_csv(
        "004_00018.top_domain_sample50.csv", index=False
    )

week3_io_timing_summary.to_csv(
    "004_00018.week3_io_timing_summary.csv", index=False
)

print("Week 3 outputs exported.")

Week 3 outputs exported.
